# **Qwen2.5-omni**

## 1.환경준비

### (1) 라이브러리 설치

In [1]:
!pip -q install -U "transformers>=4.52.0" accelerate qwen-omni-utils[decord] soundfile

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 39.2 MB/s eta 0:00:00


### (2) 라이브러리 로딩

In [2]:
import torch
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

## 2.Qwen 사용해보기

### (1) 모델 다운로드
* 모델 다운로드 : 5~8분

In [3]:
model_id = "Qwen/Qwen2.5-Omni-3B"  # 경량
model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="auto"
)
processor = Qwen2_5OmniProcessor.from_pretrained(model_id)

`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

### (2) 모델 사용하기

* 파일 준비

In [4]:
from google.colab import files
uploaded = files.upload()   # 로컬 PC에서 beach_children.jpg 선택

Saving beach_children.jpg to beach_children.jpg


* 모델 사용

In [5]:
USE_AUDIO_IN_VIDEO = False  # 영상의 내부 오디오까지 쓸지 여부
USE_AUDIO = False

conversation = [
    {"role":"system","content":[{"type":"text","text":"You are a helpful assistant."}]},
    {"role":"user","content":[
        {"type":"image","image":"beach_children.jpg"},
        {"type":"text","text":"사람 수와 상황을 설명해줘."}
    ]}
]

text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
audios, images, videos = process_mm_info(conversation, use_audio_in_video=USE_AUDIO_IN_VIDEO)

inputs = processor(
    text=text, audio=audios, images=images, videos=videos,
    return_tensors="pt", padding=True, use_audio_in_video=USE_AUDIO_IN_VIDEO
).to(model.device)

gen_kwargs = dict(
    **inputs,
    return_audio=USE_AUDIO,
    use_audio_in_video=False,
    max_new_tokens=256,
)

with torch.inference_mode():
    out = model.generate(**gen_kwargs)

# 출력 분기 처리
if USE_AUDIO:
    text_ids, audio = out            # 음성까지 요청한 경우: (text_ids, audio) 튜플
else:
    text_ids = out                   # 텍스트만 요청한 경우: 텐서 1개

print(processor.batch_decode(text_ids, skip_special_tokens=True)[0])

system
You are a helpful assistant.
user
사람 수와 상황을 설명해줘.
assistant
이 사진에는 총 3명의 사람이 보입니다. 

1. 왼쪽에 앉아 있는 두 명의 어린이가 있습니다. 그들은 모래를 모으거나 놀고 있는 것 같습니다.
2. 오른쪽에 서 있는 어린이가 있습니다. 그는 물에 담긴 물통을 들고 있으며, 바다로 향하고 있는 것 같습니다.

이들은 모두 해변에서 놀고 있는 상황입니다.


### (3) 실습
* 다양한 이미지를 다운받아, 모델에 입력하고, 사용해 봅시다.